# RAG Retrieval Experiment

Evaluate dense, sparse, and hybrid retrieval strategies for pruning
14k+ ICD10h codes down to a manageable candidate set before LLM reranking.

Dataset: Belgian SOSA 1920â€“1930 cause-of-death records.

In [23]:
# Cell 1: Imports + Config
from __future__ import annotations

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import re
import sys
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Make project root importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from codllm.config import DataSourceConfig
from codllm.data_handler import BELGIUM_MAPPING, load_source_dataset
from codllm import icd10h_registry

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Paths ---
MASTERLIST_PATH = PROJECT_ROOT / "data" / "ICD10h_Masterlist_2024.xlsx"

# --- Parameters ---
K_VALUES = [25, 50, 100, 200]
DENSE_MODEL = "intfloat/multilingual-e5-large"
SEED = 42
np.random.seed(SEED)

print(f"Device:       {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else ""))
print(f"Project root: {PROJECT_ROOT}")
print(f"K values:     {K_VALUES}")
print(f"Model:        {DENSE_MODEL}")

Device:       cuda (NVIDIA GeForce RTX 5070 Laptop GPU)
Project root: c:\Users\edlun\Desktop\DTU\Bachelor\codLLM
K values:     [25, 50, 100, 200]
Model:        intfloat/multilingual-e5-large


In [24]:
# Cell 2: Load Belgian data via existing data handler
belgium_source = DataSourceConfig(
    source_id="belgium_1920_1930",
    path="SOSA_EXTR_1920-1930 (belgium).xlsx",
    mapping_id="belgium",
)

df_belgium = load_source_dataset(
    source=belgium_source,
    mapping=BELGIUM_MAPPING,
    training_input=["cod"],
    max_labels=6,
    data_raw_dir=str(PROJECT_ROOT / "data" / "raw"),
)

# Strip the "cod: " prefix to get raw query text
df_belgium["query"] = df_belgium["text"].str.replace(r"^cod:\s*", "", regex=True)
# Gold code sets
df_belgium["gold_set"] = df_belgium["y_codes"].apply(set)

# Subsample 10% for faster iteration
SAMPLE_FRAC = 0.10
df_belgium = df_belgium.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)

queries = df_belgium["query"].tolist()
gold_sets = df_belgium["gold_set"].tolist()
record_ids = df_belgium["record_id"].tolist()

print(f"Belgian dataset (10% sample): {len(df_belgium)} records")
print(f"Gold codes per record: min={df_belgium['y_codes'].str.len().min()}, "
      f"max={df_belgium['y_codes'].str.len().max()}, "
      f"mean={df_belgium['y_codes'].str.len().mean():.2f}")
df_belgium[["record_id", "query", "y_codes"]].head()

Belgian dataset (10% sample): 4212 records
Gold codes per record: min=1, max=4, mean=1.17


,record_id,query,y_codes
0,920A0220,Péritonite généralisée,[K65.900]
1,923A1622,Aangeboren zwakte,[P96.901]
2,925A1200,Pneumonie hypostatique,[J18.200]
3,924A3871,Hémorrhagie cérébrale,[I61.900]
4,923A0059,Convulsions,[R56.800]


In [25]:
# Cell 3: Load masterlist + build corpus
masterlist_df = pd.read_excel(MASTERLIST_PATH, sheet_name="Masterlist", engine="openpyxl")
print(f"Masterlist columns: {masterlist_df.columns.tolist()}")
print(f"Masterlist rows: {len(masterlist_df)}")

# Populate the registry
valid_codes = icd10h_registry.load_masterlist(MASTERLIST_PATH)
icd10h_registry.set_valid_codes(valid_codes)
print(f"Valid ICD10h codes in registry: {len(valid_codes)}")

# Build corpus: one document per ICD10h code
# Include all descriptive columns for maximum retrieval signal
corpus_codes: list[str] = []
corpus_texts: list[str] = []

for _, row in masterlist_df.iterrows():
    code = str(row["ICD10h"]).strip()
    if not re.match(r"^[A-Z]\d{2}\.\d{3}$", code):
        continue
    desc = str(row.get("ICD10h_DESCRIPTION", "")).strip()
    cat = str(row.get("icd10_2levelCATEGORY", "")).strip()
    cause = str(row.get("ICD10_2levelCAUSE", "")).strip()
    histcat = str(row.get("HistCat", "")).strip()

    parts = [f"{code}: {desc}"]
    if cause and cause != "nan" and cause != desc:
        parts.append(f"Cause: {cause}")
    if cat and cat != "nan":
        parts.append(f"Category: {cat}")
    if histcat and histcat != "nan":
        parts.append(f"HistCat: {histcat}")
    corpus_codes.append(code)
    corpus_texts.append(" | ".join(parts))

code_to_idx = {c: i for i, c in enumerate(corpus_codes)}
idx_to_code = {i: c for i, c in enumerate(corpus_codes)}
print(f"Corpus size: {len(corpus_texts)} codes")
print(f"Example corpus entries:")
for i in range(min(5, len(corpus_texts))):
    print(f"  {corpus_texts[i]}")

# Sanity check: verify gold codes exist in corpus
all_gold = set()
for gs in gold_sets:
    all_gold.update(gs)
missing = all_gold - set(corpus_codes)
print(f"\nUnique gold codes: {len(all_gold)}")
print(f"Gold codes missing from corpus: {len(missing)}")
if missing:
    print(f"  Missing codes: {sorted(missing)[:20]}")

Masterlist columns: ['IDMasterlist', 'ICD10h', 'ICD10', 'icd10_2levelCATEGORY', 'ICD10_2levelCAUSE', 'ICD10h_DESCRIPTION', 'HistCat', 'DoNotUse', 'NotForUnderlying', 'GenderSpecific']
Masterlist rows: 14088
Valid ICD10h codes in registry: 14088
Corpus size: 14088 codes
Example corpus entries:
  A00.000: Cholera due to Vibrio cholerae 01, biovar cholerae | Category: Cholera | HistCat: Diarrhoea
  A00.100: Cholera due to Vibrio cholerae 01, biovar el tor | Category: Cholera | HistCat: Diarrhoea
  A00.900: Cholera, unspecified | Category: Cholera | HistCat: Diarrhoea
  A00.901: Choleraic diarrhoea | Cause: Cholera, unspecified | Category: Cholera | HistCat: Diarrhoea
  A01.000: Typhoid fever | Category: Typhoid and paratyphoid fevers | HistCat: Diarrhoea

Unique gold codes: 472
Gold codes missing from corpus: 7
  Missing codes: ['A09.008', 'A09.009', 'R99.010', 'S87.900', 'T58.800', 'T58.900', 'X71.001']


In [26]:
# Cell 4: Dense Retriever (E5-large only)


class DenseRetriever:
    """Dense retrieval using sentence-transformers + FAISS."""

    def __init__(
        self,
        model_name: str,
        batch_size: int = 64,
        device: str | None = None,
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.is_e5 = "e5" in model_name.lower()
        self.model = SentenceTransformer(model_name, device=device)
        self.index: faiss.IndexFlatIP | None = None
        self.corpus_embeddings: np.ndarray | None = None

    def index_corpus(self, texts: list[str]) -> float:
        """Encode corpus and build FAISS index. Returns encoding time in seconds."""
        corpus_input = [f"passage: {t}" for t in texts] if self.is_e5 else texts
        start = time.time()
        self.corpus_embeddings = self.model.encode(
            corpus_input,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        encode_time = time.time() - start

        # Build FAISS inner-product index
        dim = self.corpus_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.corpus_embeddings.astype(np.float32))
        return encode_time

    def query_batch(
        self, texts: list[str], k: int
    ) -> list[list[tuple[int, float]]]:
        """Retrieve top-k for a batch of queries."""
        query_input = [f"query: {t}" for t in texts] if self.is_e5 else texts
        q_embs = self.model.encode(
            query_input,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype(np.float32)
        scores, indices = self.index.search(q_embs, k)
        results = []
        for i in range(len(texts)):
            results.append([(int(indices[i, j]), float(scores[i, j])) for j in range(k)])
        return results


print("DenseRetriever defined.")

DenseRetriever defined.


In [27]:
# Cell 5: Index E5-large corpus
print("Indexing E5-large...")
e5_retriever = DenseRetriever(DENSE_MODEL, device=DEVICE)
encode_time = e5_retriever.index_corpus(corpus_texts)
print(f"Corpus encode time: {encode_time:.1f}s")

Indexing E5-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1003.13it/s, Materializing param=pooler.dense.weight]                              
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 221/221 [01:16<00:00,  2.90it/s]

Corpus encode time: 76.4s


In [ ]:
# Cell 6: Evaluation functions


def coverage_at_k(gold: set[str], candidates: list[str]) -> float:
    """Fraction of gold codes found in the candidate list."""
    if not gold:
        return 1.0
    return len(gold & set(candidates)) / len(gold)


def all_in_at_k(gold: set[str], candidates: list[str]) -> bool:
    """True if all gold codes are in the candidate list."""
    return gold.issubset(set(candidates))


def f1_per_record(gold: set[str], candidates: list[str]) -> dict:
    """Compute precision, recall, F1 for a single record.

    Treats as multi-label: each code is 'retrieved' or not, 'gold' or not.
    TP = gold codes found in candidates
    FP = candidates not in gold
    FN = gold codes not in candidates
    """
    cand_set = set(candidates)
    tp = len(gold & cand_set)
    fp = len(cand_set - gold)
    fn = len(gold - cand_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


def evaluate_retriever(
    retriever,
    queries: list[str],
    gold_sets: list[set[str]],
    k_values: list[int],
    idx_to_code: dict[int, str],
) -> tuple[pd.DataFrame, list[dict]]:
    """Evaluate a retriever across multiple K values.

    Returns (summary_df, per_record_results).
    """
    max_k = max(k_values)
    start = time.time()
    results = retriever.query_batch(queries, max_k)
    query_time = time.time() - start

    per_record: list[dict] = []
    summary_rows: list[dict] = []

    for k in k_values:
        coverages = []
        all_ins = []
        # Accumulators for micro-F1
        total_tp, total_fp, total_fn = 0, 0, 0
        record_f1s = []

        for i, (gold, result) in enumerate(zip(gold_sets, results)):
            candidate_codes = [idx_to_code[idx] for idx, _ in result[:k]]
            cov = coverage_at_k(gold, candidate_codes)
            ai = all_in_at_k(gold, candidate_codes)
            f1_info = f1_per_record(gold, candidate_codes)

            coverages.append(cov)
            all_ins.append(ai)
            record_f1s.append(f1_info["f1"])
            total_tp += f1_info["tp"]
            total_fp += f1_info["fp"]
            total_fn += f1_info["fn"]

            per_record.append({
                "record_idx": i,
                "k": k,
                "coverage": cov,
                "all_in": ai,
                "n_gold": len(gold),
                "precision": f1_info["precision"],
                "recall": f1_info["recall"],
                "f1": f1_info["f1"],
            })

        # Micro F1: aggregate TP/FP/FN then compute
        micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
        micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0.0

        # Macro F1: average per-record F1
        macro_f1 = np.mean(record_f1s)

        coverages_arr = np.array(coverages)
        all_ins_arr = np.array(all_ins)
        summary_rows.append({
            "K": k,
            "Mean_Cov": coverages_arr.mean(),
            "Median_Cov": np.median(coverages_arr),
            "Pct_Zero": (coverages_arr == 0).mean() * 100,
            "Pct_Partial": ((coverages_arr > 0) & (coverages_arr < 1)).mean() * 100,
            "Pct_Full": (coverages_arr == 1.0).mean() * 100,
            "AllIn_Pct": all_ins_arr.mean() * 100,
            "Micro_F1": micro_f1,
            "Macro_F1": macro_f1,
        })

    return pd.DataFrame(summary_rows), per_record, query_time

print("Evaluation functions defined.")

In [ ]:
# Cell 7: Run E5-large evaluation
print("Evaluating E5-large across K values...")
start = time.time()
summary_df, per_record_list, query_time = evaluate_retriever(
    e5_retriever, queries, gold_sets, K_VALUES, idx_to_code
)
print(f"Query time: {query_time:.1f}s")

summary_df["Model"] = "E5-large"
per_record_df = pd.DataFrame(per_record_list)
per_record_df["model"] = "E5-large"

# --- Results ---
print("\n" + "=" * 70)
print("E5-large Dense Retrieval Results")
print("=" * 70)
display(summary_df[["K", "Mean_Cov", "Median_Cov", "Pct_Zero", "Pct_Full", "AllIn_Pct", "Micro_F1", "Macro_F1"]].round(4))

# --- Per-gold-size breakdown ---
print("\n" + "=" * 70)
print("Coverage@K Breakdown by Gold-Set Size")
print("=" * 70)

per_record_df["gold_group"] = per_record_df["n_gold"].apply(
    lambda n: f"{n}" if n < 5 else "5+"
)

group_counts = per_record_df[per_record_df["k"] == K_VALUES[0]].groupby("gold_group").size()
print(f"\nRecords per gold-size group:")
print(group_counts.to_string())

pivot = per_record_df.pivot_table(
    index="gold_group",
    columns="k",
    values="coverage",
    aggfunc="mean",
)
print(f"\nMean coverage by gold-set size and K:")
display(pivot.round(4))

In [ ]:
# Cell 7b: Inclusion Rate Graph (AllIn% vs K)
import matplotlib.pyplot as plt

# Use a denser K range for a smooth curve
K_GRAPH = [5, 10, 25, 50, 75, 100, 150, 200]
max_k_graph = max(K_GRAPH)

# Retrieve top-max_k_graph for all queries (reuse existing retriever)
graph_results = e5_retriever.query_batch(queries, max_k_graph)

allin_pcts = []
for k in K_GRAPH:
    all_in_count = 0
    for gold, result in zip(gold_sets, graph_results):
        candidate_codes = {idx_to_code[idx] for idx, _ in result[:k]}
        if gold.issubset(candidate_codes):
            all_in_count += 1
    allin_pcts.append(all_in_count / len(gold_sets) * 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_GRAPH, allin_pcts, marker="o", linewidth=2, markersize=6, color="#2563eb")
ax.set_xlabel("K (number of retrieved candidates)", fontsize=12)
ax.set_ylabel("Inclusion Rate — AllIn@K (%)", fontsize=12)
ax.set_title("Inclusion Rate vs K  (E5-large, full ICD10h codes)", fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xticks(K_GRAPH)
for x, y in zip(K_GRAPH, allin_pcts):
    ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

In [30]:
# Cell 8: Example inspection
rng = np.random.RandomState(SEED)
sample_indices = rng.choice(len(queries), size=min(5, len(queries)), replace=False)
sample_queries = [queries[i] for i in sample_indices]
sample_results = e5_retriever.query_batch(sample_queries, 10)

for j, idx in enumerate(sample_indices):
    print("=" * 80)
    print(f"Record {record_ids[idx]}")
    print(f"Query: {queries[idx]}")
    print(f"Gold codes: {sorted(gold_sets[idx])}")
    print(f"\nTop-10 retrieved (E5-large):")
    for rank, (cidx, score) in enumerate(sample_results[j], 1):
        code = idx_to_code[cidx]
        in_gold = "*" if code in gold_sets[idx] else " "
        print(f"  {rank:>2d}. [{in_gold}] {corpus_texts[cidx]:<80s} (score={score:.4f})")
    print()

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.47it/s]

Record 928A0805
Query: Etter pleuris, streptokokken
Gold codes: ['A49.100', 'J86.901']

Top-10 retrieved (E5-large):
   1. [ ] A40.300: Sepsis due to Streptococcus pneumoniae | Cause: sepsis due to Streptococcus pneumoniae | Category: Streptococcal sepsis | HistCat: Infectious (score=0.8443)
   2. [ ] A40.900: Streptococcal sepsis, unspecified | Category: Streptococcal sepsis | HistCat: Infectious (score=0.8424)
   3. [ ] J02.000: Streptococcal pharyngitis | Category: Acute pharyngitis | HistCat: Respiratory (score=0.8420)
   4. [ ] A40.301: Septicaemia, pneumococcal | Cause: sepsis due to Streptococcus pneumoniae | Category: Streptococcal sepsis | HistCat: Infectious (score=0.8415)
   5. [ ] J13.000: Pneumonia due to Streptococcus pneumoniae | Category: Pneumonia due to Streptococcus pneumoniae | HistCat: Respiratory (score=0.8408)
   6. [ ] A40.901: Septicaemia, streptococcal unspecified | Cause: Streptococcal sepsis, unspecified | Category: Streptococcal sepsis | HistCat: Infectious

In [31]:
# Cell 9: Speed stats
print("Speed Statistics")
print(f"  Corpus encode: {encode_time:.1f}s")
print(f"  Query time ({len(queries)} queries): {query_time:.1f}s")
print(f"  Avg per query: {query_time / len(queries) * 1000:.1f}ms")

Speed Statistics
  Corpus encode: 76.4s
  Query time (4212 queries): 6.4s
  Avg per query: 1.5ms


In [32]:
# Cell 10: Save artifacts
OUT_DIR = PROJECT_ROOT / "data" / "cache" / "rag"
OUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_csv = OUT_DIR / "comparison_results.csv"
summary_df.to_csv(comparison_csv, index=False)
print(f"Saved comparison table: {comparison_csv}")

per_record_pkl = OUT_DIR / "per_record_results.pkl"
per_record_df.to_pickle(per_record_pkl)
print(f"Saved per-record results: {per_record_pkl}")

Saved comparison table: c:\Users\edlun\Desktop\DTU\Bachelor\codLLM\data\cache\rag\comparison_results.csv
Saved per-record results: c:\Users\edlun\Desktop\DTU\Bachelor\codLLM\data\cache\rag\per_record_results.pkl
